# Домашнее задание 1: Тренировочный цикл и *linear probe* на ViT-Tiny

### Цель

Научиться строить тренировочный цикл в PyTorch, проводить sanity-checks, работать с предобученными моделями и сравнивать простую CNN с линейным классификатором (*linear probe*) поверх ViT-Tiny.

---

### Задание

1. **Подготовка данных**

   * Соберите или выберите мини-датасет (не менее 5 классов, ≥100 изображений на класс).
   * Разбейте на `train/val`, используйте формат `ImageFolder`.
   * Реализуйте базовые аугментации: `RandomResizedCrop`, `RandomHorizontalFlip`, нормализацию.


In [53]:
import kagglehub

# Download latest version
kaggle = kagglehub.dataset_download("muratkokludataset/grapevine-leaves-image-dataset")
kaggle = kaggle + "/Grapevine_Leaves_Image_Dataset"
print(kaggle)

Using Colab cache for faster access to the 'grapevine-leaves-image-dataset' dataset.
/kaggle/input/grapevine-leaves-image-dataset/Grapevine_Leaves_Image_Dataset


In [52]:
!rm -rf "./processed"

In [54]:
import os
import shutil
from sklearn.model_selection import train_test_split

def split_dataset(raw_data_path, output_path, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15):
    for split in ['train', 'val', 'test']:
        for class_name in os.listdir(raw_data_path):
            class_path = os.path.join(raw_data_path, class_name)
            if os.path.isdir(class_path):
              os.makedirs(os.path.join(output_path, split, class_name), exist_ok=True)

    for class_name in os.listdir(raw_data_path):
        class_path = os.path.join(raw_data_path, class_name)

        if os.path.isdir(class_path):
          images = [f for f in os.listdir(class_path) if os.path.isfile(os.path.join(class_path, f))]

          train_val_images, test_images = train_test_split(
              images, test_size=test_ratio, random_state=42
          )
          train_images, val_images = train_test_split(
              train_val_images, test_size=val_ratio/(train_ratio+val_ratio), random_state=42
          )

          for img in train_images:
              src = os.path.join(class_path, img)
              dst = os.path.join(output_path, 'train', class_name, img)
              shutil.copy2(src, dst)

          for img in val_images:
              src = os.path.join(class_path, img)
              dst = os.path.join(output_path, 'val', class_name, img)
              shutil.copy2(src, dst)

          for img in test_images:
              src = os.path.join(class_path, img)
              dst = os.path.join(output_path, 'test', class_name, img)
              shutil.copy2(src, dst)

result_path = './processed'
train_dir=result_path+'/train'
val_dir=result_path+'/val'

split_dataset(kaggle, result_path)

In [61]:

import torch
from torchvision import datasets, transforms

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

# Аугментации для тренировочных данных
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Преобразования для валидации
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Загрузка данных
train_dataset = datasets.ImageFolder(
    train_dir,
    transform=train_transform
)
val_dataset = datasets.ImageFolder(
    val_dir,
    transform=val_transform
)

# Создание DataLoader
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4
)
val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4
)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


2. **Тренировочный цикл (CNN)**

   * Реализуйте простую CNN (2–3 свёрточных слоя + классификатор).
   * Добавьте фиксированные сиды, sanity-check (*overfit на нескольких батчах*).
   * Логируйте процесс в TensorBoard (лосс, accuracy, learning rate, гистограммы весов/градиентов).

In [62]:
import torch.nn as nn

class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(64 * 28 * 28, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

In [63]:
class_names = []
for class_name in os.listdir("./processed/train"):
  class_names.append(class_name)

num_classes = 5

In [72]:
import torch
import torch.nn as nn
from torch.utils.tensorboard import SummaryWriter
from torch.optim.lr_scheduler import StepLR
import numpy as np
from sklearn.metrics import confusion_matrix, f1_score
import matplotlib.pyplot as plt
import io
import torchvision
from torch.profiler import profile, record_function, ProfilerActivity

class TensorBoardLogger:
    def __init__(self, log_dir):
        self.writer = SummaryWriter(log_dir=log_dir)

    def log_scalars(self, phase, epoch, loss, accuracy, lr=None):
        """Логирование скалярных значений"""
        self.writer.add_scalar(f'Loss/{phase}', loss, epoch)
        self.writer.add_scalar(f'Accuracy/{phase}', accuracy, epoch)
        if lr is not None:
            self.writer.add_scalar('Learning_rate', lr, epoch)

    def log_histograms(self, model, epoch):
        """Логирование гистограмм весов и градиентов"""
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.writer.add_histogram(f'weights/{name}', param, epoch)
                if param.grad is not None:
                    self.writer.add_histogram(f'gradients/{name}', param.grad, epoch)

    def log_confusion_matrix(self, all_preds, all_labels, class_names, epoch):
        """Логирование confusion matrix"""
        cm = confusion_matrix(all_labels, all_preds)

        fig = plt.figure(figsize=(10, 10))
        plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
        plt.title('Confusion Matrix')
        plt.colorbar()
        tick_marks = np.arange(len(class_names))
        plt.xticks(tick_marks, class_names, rotation=45)
        plt.yticks(tick_marks, class_names)

        thresh = cm.max() / 2.
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                plt.text(j, i, format(cm[i, j], 'd'),
                        horizontalalignment="center",
                        color="white" if cm[i, j] > thresh else "black")

        plt.tight_layout()
        plt.ylabel('True label')
        plt.xlabel('Predicted label')

        buf = io.BytesIO()
        plt.savefig(buf, format='png')
        buf.seek(0)
        image = torchvision.transforms.ToTensor()(plt.imread(buf))

        self.writer.add_image('Confusion Matrix', image, epoch)
        plt.close(fig)

    def close(self):
        self.writer.close()

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs, class_names, device):
    """Основная функция тренировки с логированием"""
    logger = TensorBoardLogger('runs/experiment_1')
    best_acc = 0.0

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for i, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            # Логирование батча (каждые 50 батчей)
            if i % 50 == 0:
                batch_acc = 100 * (predicted == labels).sum().item() / labels.size(0)
                logger.log_scalars('train_batch', epoch * len(train_loader) + i,
                                  loss.item(), batch_acc,
                                  optimizer.param_groups[0]['lr'])

        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        train_loss = running_loss / len(train_loader)
        train_acc = 100 * correct / total
        val_loss = val_loss / len(val_loader)
        val_acc = 100 * val_correct / val_total
        val_f1 = f1_score(all_labels, all_preds, average='macro')

        current_lr = optimizer.param_groups[0]['lr']
        logger.log_scalars('train_epoch', epoch, train_loss, train_acc, current_lr)
        logger.log_scalars('val_epoch', epoch, val_loss, val_acc)
        logger.log_histograms(model, epoch)
        logger.log_confusion_matrix(all_preds, all_labels, class_names, epoch)

        if scheduler is not None:
            scheduler.step()

        # Сохранение лучшей модели
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), 'best_model.pth')

        print(f'Epoch [{epoch+1}/{num_epochs}], '
              f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%, '
              f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%, '
              f'Val F1: {val_f1:.4f}, LR: {current_lr:.6f}')

    logger.close()
    return best_acc

def train_cnn():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    model = SimpleCNN(num_classes=len(class_names)).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    scheduler = StepLR(optimizer, step_size=10, gamma=0.1)

    best_acc = train_model(
        model, train_loader, val_loader, criterion,
        optimizer, scheduler, num_epochs=50,
        class_names=class_names, device=device
    )

    print(f'Best validation accuracy: {best_acc:.2f}%')

def train_vit_tiny():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    model = timm.create_model('vit_tiny_patch16_224', pretrained=True)

    for param in model.parameters():
        param.requires_grad = False

    num_features = model.head.in_features
    model.head = nn.Linear(num_features, len(class_names))
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.head.parameters(), lr=0.001)
    scheduler = StepLR(optimizer, step_size=10, gamma=0.1)

    best_acc = train_model(
        model, train_loader, val_loader, criterion,
        optimizer, scheduler, num_epochs=30,
        class_names=class_names, device=device
    )

    print(f'Best validation accuracy: {best_acc:.2f}%')

def profile_model(model, train_loader, device):
    model.eval()

    with profile(
        activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
        schedule=torch.profiler.schedule(wait=1, warmup=1, active=50),
        on_trace_ready=torch.profiler.tensorboard_trace_handler('./log/profile'),
        record_shapes=True
    ) as prof:
        for step, (inputs, labels) in enumerate(train_loader):
            if step >= 100:
                break
            inputs = inputs.to(device)
            with record_function("model_inference"):
                outputs = model(inputs)
            prof.step()

In [73]:
train_cnn()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Epoch [1/50], Train Loss: 2.0304, Train Acc: 20.58%, Val Loss: 1.6041, Val Acc: 20.00%, Val F1: 0.0667, LR: 0.001000


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Epoch [2/50], Train Loss: 1.6127, Train Acc: 19.71%, Val Loss: 1.6057, Val Acc: 20.00%, Val F1: 0.0667, LR: 0.001000


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


...

Epoch [48/50], Train Loss: 1.2669, Train Acc: 50.14%, Val Loss: 1.2618, Val Acc: 51.25%, Val F1: 0.5052, LR: 0.000000


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Epoch [49/50], Train Loss: 1.3096, Train Acc: 46.09%, Val Loss: 1.2617, Val Acc: 51.25%, Val F1: 0.5052, LR: 0.000000


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Epoch [50/50], Train Loss: 1.3055, Train Acc: 46.09%, Val Loss: 1.2617, Val Acc: 51.25%, Val F1: 0.5052, LR: 0.000000
Best validation accuracy: 51.25%


In [74]:
train_vit_tiny()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Epoch [1/30], Train Loss: 1.9668, Train Acc: 19.13%, Val Loss: 1.6982, Val Acc: 30.00%, Val F1: 0.2873, LR: 0.001000


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Epoch [2/30], Train Loss: 1.5964, Train Acc: 30.43%, Val Loss: 1.4828, Val Acc: 33.75%, Val F1: 0.3294, LR: 0.001000


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


...

Epoch [28/30], Train Loss: 0.6960, Train Acc: 78.55%, Val Loss: 0.7689, Val Acc: 71.25%, Val F1: 0.7136, LR: 0.000010


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Epoch [29/30], Train Loss: 0.7436, Train Acc: 73.04%, Val Loss: 0.7680, Val Acc: 71.25%, Val F1: 0.7136, LR: 0.000010


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Epoch [30/30], Train Loss: 0.7750, Train Acc: 71.59%, Val Loss: 0.7683, Val Acc: 71.25%, Val F1: 0.7136, LR: 0.000010
Best validation accuracy: 73.75%
